# 03 — 连续隐变量：线性高斯模型
沿用第 02 章的先验 × 条件分布 → 联合密度 → 边缘密度 → MLE 的流程。

现在 $x$ 是连续标量，$y$ 是二维向量。主动定义 $p(x)=\mathcal N(0,1)$ 与 $p_\theta(y|x)=\mathcal N(wx+b,R)$，其中 $w,b$ 是待学习二维向量，$R=\sigma^2I$ 固定。先验参数也固定，以便只研究连续边缘化带来的变化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
from utils import generate_observations, compare, plot_history

# 共享数据，只观测 y；不读取真实生成类别。
rng = np.random.default_rng(42)
y = generate_observations(rng)

## 1. 定义模型，暂时固定初始参数
与第 02 章一样，先用人为选定的参数计算概率量，

In [ ]:
sigma = .45
R = sigma**2 * np.eye(2)
w0 = np.array([1., .6])
b0 = np.array([0., 0.])
prior = norm(0, 1)

def conditional(x, w, b):
    # 给定隐变量 x，计算二维高斯分布的均值。
    mean = w*x + b
    return multivariate_normal(mean, R)

def joint(x, obs, w, b):
    # p(x,y) = p(x) p(y|x)
    return prior.pdf(x) * conditional(x, w, b).pdf(obs)

## 2. 沿用第 02 章的边缘化思想：数值积分
第 02 章只需枚举三个类别；连续隐变量无法逐个枚举所有实数。这里取有限区间网格，用小矩形面积求和近似积分：$p(y)\approx\sum_j p(x_j,y)\Delta x$。

In [ ]:
obs = y[0]
grid = np.linspace(-8, 8, 4001)
dx = grid[1] - grid[0]

def numerical_marginal(obs, w, b, grid):
    dx = grid[1] - grid[0]
    evidence = 0.0
    for x in grid:
        value = joint(x, obs, w, b)
        evidence += value * dx
    return evidence

evidence = numerical_marginal(obs, w0, b0, grid)
print('numerical marginal:', evidence)

## 3. 本模型的特殊结构：边缘分布有解析表达式
生成过程等价于 $y=wx+b+\varepsilon$，其中 $x\sim N(0,1)$、$\varepsilon\sim N(0,R)$ 相互独立。因此 $y$ 仍是高斯变量，均值为 $b$，协方差为 $ww^T+R$。不必实际积分即可计算边缘密度。这比数值积分更精确、更便宜，因此训练采用解析边缘分布。

In [ ]:
def marginal(w, b):
    # 解析观测分布 p(y)，
    covariance = np.outer(w, w) + R
    return multivariate_normal(b, covariance)

exact = marginal(w0, b0).pdf(obs)
print('numerical:', evidence)
print('analytic:', exact)
assert np.isclose(evidence, exact, rtol=1e-6, atol=0)

## 4. 完全沿用第 02 章的梯度 MLE
目标仍为 $\max_{w,b}\sum_i\log p_{w,b}(y_i)$。这里只替换计算边缘密度的方法。事实上本模型也有解析参数解，但本章沿用梯度优化。

In [ ]:
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
Y = torch.tensor(y, dtype=torch.float64)
w = torch.nn.Parameter(torch.tensor(w0))
b = torch.nn.Parameter(torch.tensor(b0))
R_torch = torch.tensor(R)
opt = torch.optim.Adam([w, b], lr=.05)

def log_marginal(Y, w, b):
    # 直接构造解析边缘高斯分布；
    covariance = torch.outer(w, w) + R_torch
    distribution = torch.distributions.MultivariateNormal(b, covariance)
    # 返回每个观测的 log p(y_i)。
    return distribution.log_prob(Y)

history = []
for epoch in range(350):
    ll = log_marginal(Y, w, b).mean()
    loss = -ll
    opt.zero_grad()
    loss.backward()
    opt.step()
    history.append(ll.item())

w_mle = w.detach().numpy()
b_mle = b.detach().numpy()
print('w:', w_mle)
print('b:', b_mle)
plot_history(history)

## 5. 生成新数据
沿用第 02 章的两步采样：先采隐变量，再采条件分布。不同之处只是先验现在为连续标准正态分布。

In [ ]:
samples = []
for i in range(len(y)):
    x = prior.rvs(random_state=rng)
    distribution = conditional(x, w_mle, b_mle)
    samples.append(distribution.rvs(random_state=rng))
compare(y, np.array(samples))

## 6. 可选：后验推断（不参与 MLE）
沿用第 02 章的贝叶斯公式：$p(x|y)=p(x,y)/p(y)$。本模型还可以直接写出高斯后验，方差 $v=(1+w^Tw/\sigma^2)^{-1}$，均值 $m=v w^T(y-b)/\sigma^2$。下面只验证两种结果相同；

In [ ]:
obs = y[0]
evidence = numerical_marginal(obs, w_mle, b_mle, grid)
v = 1 / (1 + w_mle @ w_mle / sigma**2)
m = v * (w_mle @ (obs-b_mle)) / sigma**2
posterior = norm(m, np.sqrt(v))
numerical = []
analytic = []
for x in grid:
    numerical.append(joint(x, obs, w_mle, b_mle) / evidence)
    analytic.append(posterior.pdf(x))
plt.plot(grid, numerical, label='numerical')
plt.plot(grid, analytic, '--', label='analytic')
plt.legend()
plt.show()

## 小结
连续隐变量不必然使 MLE 难以计算。线性高斯结构使边缘分布仍然是单个高斯，因此也无法表达三个分离的峰。下一章将条件均值 $wx+b$ 换成神经网络：先验和噪声不变，但解析边缘通常不再存在，必须回到本章的数值积分。